In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist

tf.keras.utils.set_random_seed(42)

In [ ]:
# Ladda in Fashion MNIST dataset
# https://keras.io/api/datasets/fashion_mnist/
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

In [ ]:
# Storlek och struktur på träningsdata och testdata

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
print("Unika klasser:", np.unique(y_train))

print("Minsta pixelvärde:", X_train.min())
print("Största pixelvärde:", X_train.max())

Vi vill dela upp träningsdatan i träning och validation. Vi använder en test storlek på 10% och vi använder stratify så klassfördelningen blir ungefär lika i träning och validation.

In [ ]:
X_train_base, X_val_base, y_train_base, y_val_base = train_test_split(
    X_train,
    y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train
)

print("X_train", X_train_base.shape)
print("X_val", X_val_base.shape)
print("X_test", X_test.shape)

In [ ]:
# Lägga till en kanaldimension

X_train_base = X_train_base[..., np.newaxis]
X_val_base = X_val_base[..., np.newaxis]
X_test_base = X_test[..., np.newaxis]

print("X_train shape efter kanal dimension", X_train_base.shape)
print("X_val shape efter kanal dimension", X_val_base.shape)
print("X_test shape efter kanal dimension", X_test_base.shape)

Varje bild i datasetet har storleken 28 × 28 pixlar. En Convolutional Neural Network (CNN) förväntar sig vanligtvis att inputdata har formen (höjd, bredd, kanaler).

För färgbilder används oftast tre kanaler (röd, grön och blå). Eftersom Fashion MNIST består av gråskalebilder har varje bild endast en kanal.

Därför lägger vi till en extra dimension i datan så att varje bild får formen 28 × 28 × 1. Detta gör att datan får rätt format för CNN-modellen. 

3. Bygg en modell (CNN)

- skapa en första fungerande modell
- använda neurala nätverk (t.ex. CNN)
- säkerställa att modellen kan tränas

In [ ]:
# Grundmodell med CNN

def create_model():

    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3,3),
                      activation='relu',
                      padding='same'),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(64, (3,3),
                      activation='relu',
                      padding='same'),

        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(64, activation='relu'),

        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

4) Träna modellen

- träna modellen på träningsdata
- använda validation
- observera hur modellen utvecklas över tid

In [ ]:
model_base = create_model()

history_base = model_base.fit(
    X_train_base,
    y_train_base,
    validation_data=(X_val_base, y_val_base),
    epochs=5,
    batch_size=64
)

In [ ]:
model_base.summary()

Vi kompilerar vår baseline-modell med optimeraren Adam och en learning rate på 0.001. Adam används eftersom den är effektiv för de flesta neurala nätverk och anpassar inlärningshastigheten under träning.

Vi använder Sparse Categorical Crossentropy som loss-funktion eftersom det är ett multiklassklassificeringsproblem där labels är heltal mellan 0 och 9.

Som utvärderingsmetrik använder vi accuracy, eftersom detta är ett relativt balanserat dataset med jämnt fördelade klasser, vilket gör accuracy till ett lämpligt mått för att bedöma modellens prestanda.

In [ ]:
# Träningskurvor

def plot_history(history, title="Träningskurvor"):
    history_df = pd.DataFrame(history.history)

    plt.figure(figsize=(12,4))

    #Loss
    plt.subplot(1,2,1)
    plt.plot(history_df["loss"], label="Training loss")
    plt.plot(history_df["val_loss"], label="Validation loss")
    plt.xlabel("Epok")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.legend()

    #Accuracy
    plt.subplot(1,2,2)
    plt.plot(history_df["accuracy"], label="Training accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epok")
    plt.ylabel("Accuracy")
    plt.title("Accuracy")
    plt.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(history_base, title="Baseline (utan normalisering)")


Modellen lär sig snabbt under de första epokerna. Training loss minskar samtidigt som training accuracy ökar, vilket visar att modellen anpassar sig till träningsdatan.

Även validation loss minskar, vilket tyder på att modellen förbättrar sin generaliseringsförmåga och presterar bättre på ny data.

In [ ]:
# Utvärdera på testdata

test_loss_base, test_accuracy_base = model_base.evaluate(X_test_base, y_test, verbose=0)

print(f"Test loss: {test_loss_base:.4f}")
print(f"Test accuracy: {test_accuracy_base:.4f}")

Testresultatet är något sämre än validation-resultatet. Detta är inte ovanligt och kan bero på att validation-setet kommer från samma ursprungliga träningsdata, medan test-setet består av helt osedda bilder.

Validation-resultatet kan därför vara något mer optimistiskt, eftersom modellen indirekt har använts för att övervaka och justera träningen utifrån denna data.

Skillnaden kan även bero på naturlig variation i datan, där testsetet kan innehålla exempel som är svårare att klassificera än validation-setet. Trots detta tyder resultaten på att modellen generaliserar relativt bra.

- Modell 2 (med normalisering)

Vi normaliserar våra pixelvärden. Vi delar alla pixelvärden med 255 så att dem hamnar mellan 0 och 1 istället.

In [ ]:
# Normalisering
X_train_norm = X_train_base.astype("float32") / 255.0
X_val_norm = X_val_base.astype("float32") / 255.0
X_test_norm = X_test_base.astype("float32") / 255.0

# Kontrollera minsta och största värden efter normalisering
print("Minsta värde efter normalisering:", X_train_norm.min())
print("Största värde efter normalisering:", X_train_norm.max())

In [ ]:
model_norm = create_model()

history_norm = model_norm.fit(
    X_train_norm,
    y_train_base,
    validation_data=(X_val_norm, y_val_base),
    epochs=5,
    batch_size=64
)

In [ ]:
plot_history(history_norm, title="Normalisering (/255)")

Skillnaden mellan training accuracy och validation accuracy är relativt liten, vilket tyder på att modellen generaliserar ganska bra.

In [ ]:
# Utvärdera på testdata

test_loss_norm, test_accuracy_norm = model_norm.evaluate(
    X_test_norm,
    y_test,
    verbose=0
)

print(f"Test loss (norm): {test_loss_norm:.4f}")
print(f"Test accuracy (norm): {test_accuracy_norm:.4f}")

- Modell 3 (med standardisering)

In [ ]:
# Modell med standardisering
mean = X_train_base.mean()
std = X_train_base.std()

X_train_std = (X_train_base - mean) / std
X_val_std = (X_val_base - mean) / std
X_test_std = (X_test_base - mean) / std

In [ ]:
model_std = create_model()

history_std = model_std.fit(
    X_train_std,
    y_train_base,
    validation_data=(X_val_std, y_val_base),
    epochs=5,
    batch_size=64
)

In [ ]:
plot_history(history_std, title="Standardisering (mean/std)")

Modellen lär sig snabbt under de första epokerna. Training loss minskar samtidigt som training accuracy ökar, vilket visar att modellen anpassar sig till träningsdatan.

Även validation loss minskar i början, vilket tyder på att modellen förbättrar sin generaliseringsförmåga och presterar bättre på ny data.

Efter några epoker börjar dock validation loss plana ut, medan training loss fortsätter att minska. Detta kan vara ett tidigt tecken på överanpassning (overfitting), där modellen lär sig träningsdatan för väl.

Skillnaden mellan training accuracy och validation accuracy är dock relativt liten, vilket tyder på att modellen fortfarande generaliserar ganska bra.

In [ ]:
# Utvärdera på testdata

test_loss_std, test_accuracy_std = model_std.evaluate(
    X_test_std,
    y_test,
    verbose=0
)

print(f"Test loss (std): {test_loss_std:.4f}")
print(f"Test accuracy (std): {test_accuracy_std:.4f}")

Jämförelse

In [ ]:
results = pd.DataFrame({
    "Experiment": ["Baseline", "Normalisering", "Standardisering"],
    "Test Accuracy": [test_accuracy_base, test_accuracy_norm, test_accuracy_std],
    "Test Loss": [test_loss_base, test_loss_norm, test_loss_std]
})

results